In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "oulad_check"

student_info = pd.read_csv(RAW_DIR / "studentInfo.csv")
student_assessment = pd.read_csv(
    RAW_DIR / "studentAssessment.csv",
    dtype={"score": str}
)
assessments = pd.read_csv(
    RAW_DIR / "assessments.csv",
    dtype={"date": str}
)
student_vle = pd.read_csv(RAW_DIR / "studentVle.csv")
vle = pd.read_csv(RAW_DIR / "vle.csv")
courses = pd.read_csv(RAW_DIR / "courses.csv")
student_registration = pd.read_csv(
    RAW_DIR / "studentRegistration.csv"
)

print("OULAD files loaded successfully")

OULAD files loaded successfully


In [2]:
datasets = {
    "studentInfo": student_info,
    "studentAssessment": student_assessment,
    "assessments": assessments,
    "studentVle": student_vle,
    "vle": vle,
    "courses": courses,
    "studentRegistration": student_registration
}

for name, dataset in datasets.items():
    print(f"{name}: {dataset.shape}")

studentInfo: (32593, 12)
studentAssessment: (173912, 5)
assessments: (206, 6)
studentVle: (10655280, 6)
vle: (6364, 6)
courses: (22, 3)
studentRegistration: (32593, 5)


In [3]:
for name, dataset in datasets.items():
    print(f"\n{name}")
    print(list(dataset.columns))


studentInfo
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']

studentAssessment
['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments
['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']

studentVle
['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']

vle
['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']

courses
['code_module', 'code_presentation', 'module_presentation_length']

studentRegistration
['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']


In [4]:
for name, dataset in datasets.items():
    print(f"\n{name}")
    print(dataset.isnull().sum())


studentInfo
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

studentAssessment
id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64

assessments
code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

studentVle
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

vle
id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

courses
code_module                   0
code_p

In [5]:
for name, dataset in datasets.items():
    print(
        f"{name}: "
        f"{dataset.duplicated().sum()} exact duplicate rows"
    )

studentInfo: 0 exact duplicate rows
studentAssessment: 0 exact duplicate rows
assessments: 0 exact duplicate rows
studentVle: 787170 exact duplicate rows
vle: 0 exact duplicate rows
courses: 0 exact duplicate rows
studentRegistration: 0 exact duplicate rows


In [6]:
print("Final Result Counts:")
print(student_info["final_result"].value_counts())

print("\nFinal Result Percentages:")
print(
    student_info["final_result"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Final Result Counts:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

Final Result Percentages:
final_result
Pass           37.93
Withdrawn      31.16
Fail           21.64
Distinction     9.28
Name: proportion, dtype: float64


In [7]:
score_numeric = pd.to_numeric(
    student_assessment["score"],
    errors="coerce"
)

print("Unknown assessment scores:", score_numeric.isna().sum())

assessment_with_type = student_assessment.merge(
    assessments[["id_assessment", "assessment_type"]],
    on="id_assessment",
    how="left"
)

print("\nAssessment records by type:")
print(
    assessment_with_type["assessment_type"].value_counts()
)

print("\nUnknown scores by assessment type:")
print(
    assessment_with_type.loc[
        score_numeric.isna(),
        "assessment_type"
    ].value_counts()
)

Unknown assessment scores: 173

Assessment records by type:
assessment_type
TMA     98426
CMA     70527
Exam     4959
Name: count, dtype: int64

Unknown scores by assessment type:
assessment_type
TMA    173
Name: count, dtype: int64


In [8]:
print("VLE date range:")
print(
    student_vle["date"].min(),
    "to",
    student_vle["date"].max()
)

print("\nVLE click statistics:")
print(student_vle["sum_click"].describe())

print("\nNumber of students:")
print(student_vle["id_student"].nunique())

print("\nNumber of learning resources:")
print(student_vle["id_site"].nunique())

VLE date range:
-25 to 269

VLE click statistics:
count    1.065528e+07
mean     3.716946e+00
std      8.849047e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.977000e+03
Name: sum_click, dtype: float64

Number of students:
26074

Number of learning resources:
6268


In [9]:
print("VLE Activity Types:")
print(vle["activity_type"].value_counts())

VLE Activity Types:
activity_type
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
glossary            21
ouelluminate        21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: count, dtype: int64


In [10]:
student_info_pairs = student_info[
    [
        "id_student",
        "code_module",
        "code_presentation"
    ]
].drop_duplicates()

student_vle_pairs = student_vle[
    [
        "id_student",
        "code_module",
        "code_presentation"
    ]
].drop_duplicates()

print(
    "Student-course records in studentInfo:",
    len(student_info_pairs)
)

print(
    "Student-course records in studentVle:",
    len(student_vle_pairs)
)

print(
    "Students in studentInfo:",
    student_info["id_student"].nunique()
)

print(
    "Students in studentVle:",
    student_vle["id_student"].nunique()
)

print(
    "Course/presentation combinations in studentInfo:",
    student_info_pairs[
        ["code_module", "code_presentation"]
    ].drop_duplicates().shape[0]
)

print(
    "Course/presentation combinations in studentVle:",
    student_vle_pairs[
        ["code_module", "code_presentation"]
    ].drop_duplicates().shape[0]
)

Student-course records in studentInfo: 32593
Student-course records in studentVle: 29228
Students in studentInfo: 28785
Students in studentVle: 26074
Course/presentation combinations in studentInfo: 22
Course/presentation combinations in studentVle: 22


In [11]:
print("Student Information Preview:")
display(student_info.head())

print("\nAssessment Preview:")
display(student_assessment.head())

print("\nVLE Interaction Preview:")
display(student_vle.head())

Student Information Preview:


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass



Assessment Preview:


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79



VLE Interaction Preview:


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


# Data Inspection Summary

The OULAD dataset contains more than 10,000 real student-course records and provides information about student characteristics, assessments, course registration, and virtual learning environment engagement.

The studentInfo table contains 32,593 student-course records. The studentVle table contains more than 10 million interaction records, providing detailed engagement information.

The original final_result variable contains four outcome categories: Pass, Distinction, Fail, and Withdrawn. For this project, Fail and Withdrawn are later mapped to academic risk, while Pass and Distinction are mapped to non-risk.

Assessment scores contain a small number of unknown values represented by the character '?'. These values are handled during preprocessing.

The raw dataset is preserved without modification. Feature engineering and validation are performed by src/prepare_data.py.

The project defines day 90 as the intervention cutoff. Only information available from day 0 through day 90 is used to construct early-intervention assessment and engagement features.

The processed dataset is validated for missing values and duplicate student-course-presentation records before being split into training, validation, and test sets.